In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchaudio
from torch import nn
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2FeatureExtractor, Wav2Vec2Config
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
import librosa
import os

# Label Mapping


In [2]:
label_map = {
    'none': 0,
    'เย็ด': 1,
    'กู': 2,
    'มึง': 3,
    'เหี้ย': 4,
    'ควย': 5,
    'สวะ': 6,
    'หี': 7,
    'แตด': 8
}

# Define the number of labels
num_labels = len(label_map)

# Dataset
## Loading Dataset

In [3]:
df = pd.read_csv('csv/main.csv')
df

,file_path,start_time,end_time,label
0,./main/Janพูดถึงอาท.wav,0.000000,14.183573,none
1,./main/Janพูดถึงอาท.wav,14.183573,14.861386,เหี้ย
2,./main/Janพูดถึงอาท.wav,14.861386,16.214891,none
3,./main/Janพูดถึงอาท.wav,16.214891,16.685348,เหี้ย
4,./main/Janพูดถึงอาท.wav,16.685348,18.238890,none
...,...,...,...,...
1464,./main/ไอ้เหี้ยทำไมนิสัยมึงสวะได้ขนาดนี้.wav,3.530134,4.683388,none
1465,./main/ไอ้เหี้ยทำไมนิสัยมึงสวะได้ขนาดนี้.wav,4.683388,4.794722,มึง
1466,./main/ไอ้เหี้ยทำไมนิสัยมึงสวะได้ขนาดนี้.wav,4.794722,4.981679,none
1467,./main/ไอ้เหี้ยทำไมนิสัยมึงสวะได้ขนาดนี้.wav,4.981679,5.093013,มึง


## Data Processing

In [4]:
# Preprocessing function for each row
def preprocess_row(row, target_sr=16000):
    # Load the full audio file
    waveform, sample_rate = torchaudio.load(row['file_path'])
    # Extract segment
    start_sample = int(row['start_time'] * sample_rate)
    end_sample = int(row['end_time'] * sample_rate)
    segment = waveform[:, start_sample:end_sample]
    # Convert to mono
    if segment.shape[0] > 1:
        segment = segment.mean(dim=0, keepdim=True)
    # Resample if needed
    if sample_rate != target_sr:
        resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=target_sr)
        segment = resampler(segment)
    # Normalize
    segment = segment / segment.abs().max()
    # Map label
    label = label_map[row['label']]
    return segment.squeeze(0), label

#  preprocess the first row
segment, label = preprocess_row(df.iloc[0])
print(segment.shape, label)

torch.Size([226937]) 0


In [5]:
# Apply preprocessing to all rows
processed = [preprocess_row(row) for _, row in df.iterrows()]
segments, labels = zip(*processed)

# Convert to tensors
segments = list(segments)
labels = torch.tensor(labels)

# Optionally, pad/truncate segments to the same length for batching
from torch.nn.utils.rnn import pad_sequence
segments_padded = pad_sequence(segments, batch_first=True)

# Create a custom Dataset
from torch.utils.data import Dataset

class AudioDataset(Dataset):
    def __init__(self, segments, labels):
        self.segments = segments
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_values': self.segments[idx],
            'labels': self.labels[idx]
        }

dataset = AudioDataset(segments_padded, labels)

## Dealing With Imbalanced classes

In [6]:
# Calculate class weights for imbalanced dataset
unique_labels, counts = torch.unique(dataset.labels, return_counts=True)
class_weights = 1.0 - (counts.float() / len(dataset))
class_weights = class_weights.numpy()

print("Class distribution:", dict(zip(unique_labels.tolist(), counts.tolist())))
print("Class weights:", class_weights)

Class distribution: {0: 674, 1: 88, 2: 146, 3: 117, 4: 84, 5: 114, 6: 66, 7: 89, 8: 91}
Class weights: [0.5411845  0.9400953  0.90061265 0.920354   0.9428182  0.9223962
 0.95507145 0.93941456 0.9380531 ]


In [7]:
class_weights = torch.from_numpy(class_weights).float().to('cuda')
class_weights

tensor([0.5412, 0.9401, 0.9006, 0.9204, 0.9428, 0.9224, 0.9551, 0.9394, 0.9381],
       device='cuda:0')

## Trainer

### configuration 

In [8]:
class weightedLoss(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Feed inputs to model and extract logits
        outputs = model(**inputs)
        logits = outputs.get("logits")
        # Extract labels
        labels = inputs.get("labels")
        # Define loss function with class weights
        loss_func = nn.CrossEntropyLoss(weight=class_weights)
        # Compute loss
        loss = loss_func(logits.view(-1, num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss
    
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    accuracy = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    precision = precision_score(labels, preds, average='weighted')
    recall = recall_score(labels, preds, average='weighted')
    
    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

batch_size = 16
# Log the training loss at each epoch
logging_steps = len(dataset) // batch_size
output_dir = './output'
training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=2e-5,
    num_train_epochs=10,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    logging_steps=logging_steps,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    fp16=True,
    dataloader_pin_memory=False,
    # Add these for better progress display# Add these for better progress display
    report_to=None,  # Disable wandb/tensorboard
    disable_tqdm=False,  # Enable progress bars
    log_level="info",
    logging_first_step=True
)

### Training

In [9]:
indices = list(range(len(dataset)))
train_idx, eval_idx = train_test_split(indices, test_size=0.2, random_state=42)
train_dataset = torch.utils.data.Subset(dataset, train_idx)
eval_dataset = torch.utils.data.Subset(dataset, eval_idx)

In [10]:
model_name = "airesearch/wav2vec2-large-xlsr-53-th"
# Load pre-trained model and feature extractor
config = Wav2Vec2Config.from_pretrained(
    model_name,
    num_labels=num_labels,
    finetuning_task="audio-classification"
)
    # Initialize the model with default classifier
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    model_name,
    config=config,
)

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
    model_name,
    return_attention_mask=True,
    do_normalize=True,
)


c:\Users\muldi\Documents\Playground\University\senior-project-1\env\lib\site-packages\transformers\configuration_utils.py:312: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at airesearch/wav2vec2-large-xlsr-53-th and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Safetensors PR exists


In [11]:
trainer = weightedLoss(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics
)

Using auto half precision backend


In [ ]:
# Add this before training
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Add this before training
from transformers.utils import logging
from tqdm.auto import tqdm
from tqdm.notebook import tqdm

# Enable transformers logging
logging.set_verbosity_info()

# Force enable tqdm progress bars for Jupyter
os.environ["DISABLE_TQDM"] = "0"
tqdm.pandas()

# Set environment variables for memory management
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Clear GPU cache
torch.cuda.empty_cache()

print("Starting training...")
print(f"Dataset size: {len(train_dataset)}")
print(f"Batch size: {batch_size}")
print(f"Total epochs: 10")
print(f"Steps per epoch: {len(train_dataset) // batch_size}")

trainer.train()

***** Running training *****
  Num examples = 1,175
  Num Epochs = 10
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 1
  Total optimization steps = 740
  Number of trainable parameters = 315,703,433


Starting training...
Dataset size: 1175
Batch size: 16
Total epochs: 10
Steps per epoch: 73


Epoch,Training Loss,Validation Loss
